# MTG Autoregressive Deck Constructor — Inference Notebook

Generate complete 60-card decklists from archetype identity using the trained model.

Archetype names are format-prefixed (e.g., `standard::Azorius Control`, `pioneer::Rakdos Vampires`).

**Workflow:**
1. Load a trained model checkpoint
2. Pick an archetype (with format prefix)
3. Run greedy decode — the model autoregressively builds a full 60-card deck

**Runtime**: T4 GPU recommended but CPU works fine for inference.

## 1. Setup

In [ ]:
# Mount Google Drive (project lives here)
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
import sys

PROJECT_DIR = '/content/drive/MyDrive/mtg-graph'
os.chdir(PROJECT_DIR)

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

print(f'Working directory: {os.getcwd()}')

In [ ]:
# Install third-party dependencies only
!pip install -q torch-geometric sentence-transformers python-dotenv \
    beautifulsoup4 lxml spacy tqdm pyarrow

## 2. Load Graph & Model

Point `MODEL_SUBDIR` to a results directory containing a model checkpoint.

In [ ]:
import json
from pathlib import Path

import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np

from src.config import GRAPH_PATH, CARDS_PARQUET, PROJECT_ROOT, RESULTS_DIR
from src.deck_predictor import MTGDeckModel

# ── Configure model path ──
MODEL_SUBDIR = 'results/deck/latest'

MODEL_DIR = PROJECT_ROOT / MODEL_SUBDIR
model_path = None
for name in ['model_fold0.pt', 'model.pt']:
    p = MODEL_DIR / name
    if p.exists():
        model_path = p
        break
assert model_path is not None, f'No model checkpoint found in {MODEL_DIR}'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Load graph
data = torch.load(GRAPH_PATH, weights_only=False)
data = data.to(device)

# Load checkpoint and reconstruct model
checkpoint = torch.load(model_path, weights_only=False, map_location=device)
hp = checkpoint.get('hyperparameters', {})

node_dims = {nt: data[nt].x.shape[1] for nt in data.node_types}
card_names = data['card'].names
arch_names = data['archetype'].names

# Extract card metadata for budget signal if the model was trained with it
card_color_flags = None
card_type_flags = None
if hp.get('budget_signal', False):
    emb_dim = 384
    card_type_flags = data['card'].x[:, emb_dim + 1 : emb_dim + 8].clone()
    card_color_flags = data['card'].x[:, emb_dim + 10 : emb_dim + 15].clone()
    print(f'Budget signal enabled: extracted color ({card_color_flags.shape[1]}) '
          f'and type ({card_type_flags.shape[1]}) flags')

model = MTGDeckModel(
    node_dims=node_dims,
    edge_types=list(data.edge_types),
    node_types=list(data.node_types),
    card_names=card_names,
    d_model=hp.get('d_model', 128),
    d_message=hp.get('d_message', 128),
    d_count=hp.get('d_count', 16),
    num_gnn_layers=hp.get('num_gnn_layers', 2),
    dropout=0.0,
    edge_chunk_size=hp.get('edge_chunk_size', 2_000_000),
    card_color_flags=card_color_flags,
    card_type_flags=card_type_flags,
).to(device)

model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
for p in model.parameters():
    p.requires_grad = False

print(f'Model loaded from: {model_path}')
print(f'Hyperparameters: {hp}')
print(f'\nAvailable archetypes:')
for i, name in enumerate(arch_names):
    print(f'  {i}: {name}')

## 3. Select Archetype

In [ ]:
# ── Pick an archetype to build a deck for ──
# Archetype names are format-prefixed: 'standard::Name' or 'pioneer::Name'
# Format prefix options: 'standard', 'pioneer'
# See full list of available archetypes printed in the cell above.
ARCHETYPE = 'standard::Azorius Control'  # Edit to any archetype from the list above

arch_to_idx = {name: i for i, name in enumerate(arch_names)}
assert ARCHETYPE in arch_to_idx, f'Unknown archetype: "{ARCHETYPE}". See list above.'

print(f'Building deck for: {ARCHETYPE}')

## 4. Build Deck

In [ ]:
arch_idx = arch_to_idx[ARCHETYPE]

with torch.no_grad():
    deck = model.predict_deck(data, arch_idx)

# Load card metadata for display
cards_df = pd.read_parquet(CARDS_PARQUET)
card_meta = {}
for _, row in cards_df.iterrows():
    name = row['name']
    card_meta[name] = {
        'type_line': row.get('type_line', ''),
        'cmc': row.get('cmc', 0),
        'set': str(row.get('set', '')).upper(),
    }

# Group by card type for display
categories = {
    'Creatures': [], 'Instants': [], 'Sorceries': [],
    'Enchantments': [], 'Artifacts': [], 'Planeswalkers': [],
    'Lands': [], 'Other': [],
}

for card_idx, count in deck.items():
    name = card_names[card_idx]
    meta = card_meta.get(name, {})
    tl = str(meta.get('type_line', '')).lower()
    if 'creature' in tl:
        categories['Creatures'].append((name, count, meta))
    elif 'instant' in tl:
        categories['Instants'].append((name, count, meta))
    elif 'sorcery' in tl:
        categories['Sorceries'].append((name, count, meta))
    elif 'enchantment' in tl:
        categories['Enchantments'].append((name, count, meta))
    elif 'artifact' in tl:
        categories['Artifacts'].append((name, count, meta))
    elif 'planeswalker' in tl:
        categories['Planeswalkers'].append((name, count, meta))
    elif 'land' in tl:
        categories['Lands'].append((name, count, meta))
    else:
        categories['Other'].append((name, count, meta))

total = sum(deck.values())
print(f'=== {ARCHETYPE} — {total} cards ===\n')

for cat_name in ['Creatures', 'Instants', 'Sorceries', 'Enchantments',
                  'Artifacts', 'Planeswalkers', 'Lands', 'Other']:
    cards = categories[cat_name]
    if not cards:
        continue
    cards.sort(key=lambda x: (x[2].get('cmc', 0), x[0]))
    cat_total = sum(c for _, c, _ in cards)
    print(f'--- {cat_name} ({cat_total}) ---')
    for name, count, meta in cards:
        cmc = meta.get('cmc', '?')
        print(f'  {count}x {name}  (CMC {cmc})')
    print()

print(f'Total: {total} cards ({"OK" if total == 60 else "FAIL"})')

## 5. Archetype Similarity

In [ ]:
import matplotlib.pyplot as plt

with torch.no_grad():
    x_dict = model.gnn(data)
    arch_emb = x_dict['archetype']
    
    # Get the deck's card embeddings and mean-pool them
    deck_card_indices = list(deck.keys())
    deck_card_emb = x_dict['card'][deck_card_indices]
    deck_emb = deck_card_emb.mean(dim=0)
    
    similarities = F.cosine_similarity(
        deck_emb.unsqueeze(0), arch_emb, dim=1
    ).cpu().numpy()

sorted_indices = np.argsort(similarities)[::-1]
names_sorted = [arch_names[i] for i in sorted_indices]
sims_sorted = [similarities[i] for i in sorted_indices]

fig, ax = plt.subplots(figsize=(10, max(3, len(arch_names) * 0.5)))
bar_colors = ['steelblue' if s >= 0 else 'salmon' for s in sims_sorted]
ax.barh(range(len(names_sorted)), sims_sorted, color=bar_colors)
ax.set_yticks(range(len(names_sorted)))
ax.set_yticklabels(names_sorted)
ax.set_xlabel('Cosine Similarity')
ax.set_title(f'{ARCHETYPE} Deck vs All Archetypes')
ax.invert_yaxis()
ax.grid(True, alpha=0.3, axis='x')
for i, (name, sim) in enumerate(zip(names_sorted, sims_sorted)):
    ax.text(sim + 0.01, i, f'{sim:.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

print(f'Most similar archetype: {names_sorted[0]} ({sims_sorted[0]:.4f})')

## 6. Export Decklist (MTGA Import Format)

Saves the decklist in **MTG Arena import format** to `results/inference/<timestamp>/decklist.txt`.

To import into MTGA: copy the text output below, open MTGA, go to Decks, and click Import.

In [ ]:
from datetime import datetime, timezone

def mtga_line(name, count):
    meta = card_meta.get(name, {})
    set_code = meta.get('set', '')
    if set_code:
        return f'{count} {name} ({set_code})'
    return f'{count} {name}'

# Build MTGA import string
mtga_lines = []
for cat_name in ['Creatures', 'Instants', 'Sorceries', 'Enchantments',
                  'Artifacts', 'Planeswalkers', 'Lands', 'Other']:
    cards = categories[cat_name]
    if not cards:
        continue
    for name, count, _ in sorted(cards, key=lambda x: x[0]):
        mtga_lines.append(mtga_line(name, count))

mtga_text = '\n'.join(mtga_lines)

# Save
run_name = datetime.now(timezone.utc).strftime('%Y-%m-%d_%H%M%S')
out_dir = RESULTS_DIR / 'inference' / run_name
out_dir.mkdir(parents=True, exist_ok=True)
(out_dir / 'decklist.txt').write_text(mtga_text, encoding='utf-8')

print('MTGA IMPORT (copy everything below)')
print('\u2500' * 50)
print(mtga_text)
print(f'\nSaved to: {out_dir}')